In [17]:
import requests
import numpy as np
from skyfield.api import EarthSatellite, load, wgs84, utc

In [4]:
tlesource = requests.get("https://celestrak.org/NORAD/elements/gp.php?GROUP=active").text.strip().split("\n")
tles = [tlesource[i:i+3] for i in range(0, len(tlesource), 3)]
print(f"Loaded {len(tles)} TLEs")

def get_tle(sat_name):
    for tle in tles:
        if tle[0].strip() == sat_name:
            return tle
    return None

Loaded 14121 TLEs


In [43]:
def satellite_elevation(t, lat_deg, lon_deg, height_m, tle_3line):
    name, line1, line2 = tle_3line

    ts = load.timescale()
    import datetime as dt

    def _to_aware(x):
        if isinstance(x, dt.datetime):
            return x if x.tzinfo is not None else x.replace(tzinfo=utc)
        if isinstance(x, np.datetime64):
            us = x.astype('datetime64[us]').astype('int')
            return dt.datetime.utcfromtimestamp(us / 1e6).replace(tzinfo=utc)
        raise TypeError(f"Unsupported datetime type: {type(x)}")

    if isinstance(t, np.ndarray) and np.issubdtype(t.dtype, np.datetime64):
        t_py = [_to_aware(x) for x in t]
    else:
        t_py = _to_aware(t)

    t_sf = ts.from_datetime(t_py)

    satellite = EarthSatellite(line1, line2, name, ts)

    station = wgs84.latlon(
        latitude_degrees=lat_deg,
        longitude_degrees=lon_deg,
        elevation_m=height_m,
    )

    difference = satellite.at(t_sf) - station.at(t_sf)
    elevation, _, _ = difference.altaz()

    return elevation.degrees

In [ ]:
# Moscow:
lat_moscow = 55.7
lon_moscow = 37.1
height_moscow = 170
t = np.datetime64('now')
elev = satellite_elevation(t, lat_moscow, lon_moscow, height_moscow, get_tle("ELEKTRO-L 2"))
print(elev)


17.776297201362688


/var/folders/4d/1d5j55lj60v45fpt1wk0gwm80000gn/T/ipykernel_8205/3540138279.py:12: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return dt.datetime.utcfromtimestamp(us / 1e6).replace(tzinfo=utc)
